In [ ]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import torch.nn as nn

In [ ]:
class SpotifySequenceDataset(Dataset):
    def __init__(self, history_csv, master_db_csv, max_seq_length=10, mode='dpo'):
        self.max_seq_length = max_seq_length
        self.mode = mode
        
        # 1. Build the in-memory feature dictionary
        self.feature_lookup = self._build_feature_dict(master_db_csv)
        
        # 2. Process the raw history
        history_df = pd.read_csv(history_csv)
        sessions = self._chunk_sessions(history_df)
        nip_data, dpo_data = self._extract_training_data(sessions)
        
        # 3. Store only the data requested by the mode
        self.data = dpo_data if mode == 'dpo' else nip_data
        print(f"Dataset initialized in {mode.upper()} mode with {len(self.data)} samples.")

    def _build_feature_dict(self, master_db_csv):
        """Converts Master DB into a dictionary of PyTorch tensors."""
        df = pd.read_csv(master_db_csv)
        feature_dict = {}
        
        feature_cols = ['danceability', 'energy', 'key', 'loudness', 'mode', 
                        'speechiness', 'acousticness', 'instrumentalness', 
                        'liveness', 'valence', 'tempo', 'time_signature']
        
        # Convert the feature columns to a NumPy array, then to a PyTorch dictionary
        for _, row in df.iterrows():
            # Create a 1D tensor of the audio features
            features = torch.tensor(row[feature_cols].values.astype(np.float32))
            feature_dict[row['id']] = features
            
        return feature_dict

    def chunk_sessions(history_df):
        """
        Vectorized sessionization of listening history based on 
        20-minute gaps or context changes.
        """
        # 1. Ensure absolute chronological order
        df = history_df.copy()
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.sort_values('timestamp').reset_index(drop=True)
    
        # 2. Calculate the time gap between each song and the one before it
        time_gaps = df['timestamp'].diff()
    
        # 3. Check if the context_uri changed from the previous row
        context_changed = df['context_uri'] != df['context_uri'].shift(1)
    
        # 4. A new session starts IF the gap is > 20 mins OR the context changed
        is_new_session = (time_gaps > pd.Timedelta(minutes=20)) | context_changed
    
        # 5. Create a unique ID for each session using cumulative sum
        df['session_id'] = is_new_session.cumsum()
    
        # 6. Group the dataframe into a list of smaller session dataframes
        sessions = [group for _, group in df.groupby('session_id')]
    
        print(f"Divided {len(df)} tracks into {len(sessions)} distinct listening sessions.")
        return sessions
    
    def extract_training_data(sessions, max_seq_length=10):
        """
        Slides a window across each session to simultaneously extract 
        Next-Item Prediction (NIP) flows and Direct Preference Optimization (DPO) pairs.
        """
        nip_dataset = []
        dpo_dataset = []
        
        for session_df in sessions:
            current_context = []
            pending_negatives = []
            
            # Loop through the session chronologically
            for row in session_df.itertuples():
                track_id = row.track_id
                
                # 1. The Explicit Rejection (Skip)
                if row.completion_rate < 0.10:
                    pending_negatives.append(track_id)
                    
                # 2. The Good Path (Track Accepted)
                elif row.completion_rate > 0.50:
                    
                    if len(current_context) > 0:
                        
                        # NIP TARGET
                        nip_dataset.append({
                            'context': list(current_context),
                            'target': track_id
                        })
                        
                        # DPO TARGETS
                        if pending_negatives:
                            for negative_track in pending_negatives:
                                dpo_dataset.append({
                                    'context': list(current_context),
                                    'positive': track_id,
                                    'negative': negative_track
                                })
                                
                    pending_negatives = []
                    
                    current_context.append(track_id)
                    
                    if len(current_context) > max_seq_length:
                        current_context.pop(0)
                
                # 3. Mediocre Track
                else:
                    # Completion is between 10% and 50%
                    current_context = []
                    pending_negatives = []
                    
        print(f"Extracted {len(nip_dataset)} Next-Item (NIP) sequences.")
        print(f"Extracted {len(dpo_dataset)} Preference (DPO) pairs.")
        
        return nip_dataset, dpo_dataset
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Retrieves a single training example and converts it to Tensors."""
        sample = self.data[idx]
        context_ids = sample['context']
        
        # 1. Determine how many songs are actually in the context
        seq_len = len(context_ids)
        
        num_features = 12 
        
        # 2. Create the blank Left-Padded matrix (Size: 10 rows x 12 columns)
        context_tensor = torch.zeros((self.max_seq_length, num_features), dtype=torch.float32)
        
        # 3. Fill the matrix from the right side
        for i, track_id in enumerate(context_ids):
            padded_idx = self.max_seq_length - seq_len + i 
            
            if track_id in self.feature_lookup:
                context_tensor[padded_idx] = self.feature_lookup[track_id]
            else:
                context_tensor[padded_idx] = torch.full((num_features,), -1.0)
                
        # 4. Fetch the targets
        if self.mode == 'dpo':
            pos_features = self.feature_lookup.get(sample['positive'], torch.full((num_features,), -1.0))
            neg_features = self.feature_lookup.get(sample['negative'], torch.full((num_features,), -1.0))
            return context_tensor, pos_features, neg_features
            
        elif self.mode == 'nip':
            target_features = self.feature_lookup.get(sample['target'], torch.full((num_features,), -1.0))
            return context_tensor, target_features

In [ ]:
class SpotifySASRec(nn.Module):
    def __init__(self, num_features=12, d_model=128, max_seq_length=10):
        super().__init__()
        self.feature_proj = nn.Linear(num_features, d_model)
        self.pos_emb = nn.Embedding(max_seq_length, d_model)
        self.attention_block = nn.TransformerEncoderLayer(d_model=d_model, nhead=8, batch_first=True)
        
    def forward(self, context_features, causal_mask):
        # 1. Process the context sequence
        x = self.feature_proj(context_features)
        positions = torch.arange(x.size(1), device=x.device)
        x = x + self.pos_emb(positions)
        
        # 2. Extract the "Present Moment"
        transformer_out = self.attention_block(x, src_mask=causal_mask)
        current_vibe = transformer_out[:, -1, :] # Shape: [Batch, 128]
        return current_vibe
    
    def score_candidates(self, current_vibe, target_features):
        # 1. Process the candidate track
        target_vibe = self.feature_proj(target_features) # Shape: [Batch, 128]
        
        # 2. The Match: Dot product between the Context and the Target
        scores = (current_vibe * target_vibe).sum(dim=-1) 
        return scores
    
    def generate_causal_mask(seq_length):
        mask = torch.tril(torch.ones(seq_length, seq_length))
        mask = mask.masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask